# ClinicSense – Notebook 06: Dynamic Overbooking Simulator
**B.Sc. Data Science Honours | Student: Kadam Akanksha Dinkar**

**Part 2: Prescriptive Analytics**
- Monte Carlo simulation of clinic days
- Compare: No Overbooking | Fixed Overbooking | Risk-Based Dynamic Overbooking
- Cost minimisation and strategy selection
---

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.simulator import (
    compare_strategies, plot_strategy_comparison,
    cost_heatmap, monte_carlo, risk_based_overbooking
)
from src.config import (
    FIGURES_DIR, SLOTS_PER_DAY, SIM_DAYS,
    COST_IDLE_SLOT, COST_WAITING_MINUTE, COST_OVERTIME_MINUTE,
    SLOT_DURATION_MIN
)

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Setup complete.')

Setup complete.


In [2]:
# Load best model probabilities
data_dir = os.path.join('..', 'data')
best_output = joblib.load(os.path.join(data_dir, 'best_model_output.joblib'))
cal_prob = best_output['calibrated_prob']  # predicted attendance prob = 1 - no_show_prob

# Convert to attendance probabilities
attend_prob = 1 - cal_prob

print(f'Attendance probabilities computed: {len(attend_prob):,} appointments')
print(f'Mean attendance probability: {attend_prob.mean():.3f}')
print(f'Simulation cost assumptions:')
print(f'  Idle slot     : ₹{COST_IDLE_SLOT} / slot')
print(f'  Waiting       : ₹{COST_WAITING_MINUTE} / minute')
print(f'  Overtime      : ₹{COST_OVERTIME_MINUTE} / minute')
print(f'  Slot duration : {SLOT_DURATION_MIN} minutes')
print(f'  Slots per day : {SLOTS_PER_DAY}')

Attendance probabilities computed: 14,392 appointments
Mean attendance probability: 0.692
Simulation cost assumptions:
  Idle slot     : ₹150 / slot
  Waiting       : ₹2 / minute
  Overtime      : ₹5 / minute
  Slot duration : 20 minutes
  Slots per day : 20


## 1. Build Daily Appointment Pools

In [3]:
# Group appointment probabilities into simulated daily pools
# Each pool represents the booked-patient probabilities for one clinic day
# We sample SLOTS_PER_DAY * 1.5 patients per pool to allow for overbooking headroom

rng = np.random.default_rng(42)
pool_size    = int(SLOTS_PER_DAY * 2)   # enough patients to draw extras from
n_pools      = max(200, SIM_DAYS)       # pre-generate enough pools

daily_pools = [
    rng.choice(attend_prob, size=pool_size, replace=True)
    for _ in range(n_pools)
]

print(f'Generated {len(daily_pools):,} daily pools of {pool_size} patients each.')
print(f'Sample pool mean attendance prob: {daily_pools[0].mean():.3f}')

Generated 5,000 daily pools of 40 patients each.
Sample pool mean attendance prob: 0.685


## 2. Run All Overbooking Strategies

In [4]:
print(f'Running Monte Carlo simulation ({SIM_DAYS:,} days per strategy)...')
print('='*60)
summary, raw_results = compare_strategies(daily_pools,
                                           n_slots=SLOTS_PER_DAY,
                                           n_days=SIM_DAYS)

print('\n=== STRATEGY COMPARISON SUMMARY ===')
print(summary.to_string())

summary.to_csv(os.path.join('..', 'outputs', 'reports', 'simulation_summary.csv'))
print('\nSimulation summary saved.')

Running Monte Carlo simulation (5,000 days per strategy)...
  Running strategy: No Overbooking (0) … 

done
  Running strategy: Fixed +1 … 

done
  Running strategy: Fixed +2 … 

done
  Running strategy: Fixed +3 … 

done
  Running strategy: Fixed +4 … 

done
  Running strategy: Fixed +5 … 

done
  Running strategy: Risk-Based Dynamic … 

done

=== STRATEGY COMPARISON SUMMARY ===
                    Avg Total Cost  Avg Idle Slots  Avg Overflow  Avg Utilisation %  Avg Wait (min)  Avg Overtime (min)  Std Total Cost
Strategy                                                                                                                               
No Overbooking (0)          928.35            6.19          0.00              69.06            0.00                0.00          309.30
Fixed +1                    821.34            5.48          0.00              72.62            0.00                0.00          317.69
Fixed +2                    717.78            4.78          0.00              76.09            0.06                0.06          317.06
Fixed +3                    631.82            4.20          0.01              79.00            0.26                0.26          325.66
Fixed +4                    526.68            3.47          0.05              82.67            0.96                0.96          320.75
Fixed 

## 3. Visualise Strategy Comparison

In [5]:
plot_strategy_comparison(summary)
print('Strategy comparison plot saved.')

Strategy comparison plot saved.


In [6]:
# Cost distribution comparison for key strategies
key_strategies = ['No Overbooking (0)', 'Fixed +3', 'Risk-Based Dynamic']
key_strategies_avail = [s for s in key_strategies if s in raw_results]

fig, axes = plt.subplots(1, len(key_strategies_avail), figsize=(16, 5), sharey=False)
if len(key_strategies_avail) == 1:
    axes = [axes]

for ax, name in zip(axes, key_strategies_avail):
    df_s = raw_results[name]
    ax.hist(df_s['total_cost'], bins=60, color='steelblue', alpha=0.8,
             edgecolor='white', density=True)
    ax.axvline(df_s['total_cost'].mean(), color='red', linestyle='--',
                linewidth=2, label=f'Mean: ₹{df_s["total_cost"].mean():.0f}')
    ax.set_title(f'{name}')
    ax.set_xlabel('Total Daily Cost (₹)')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Daily Cost Distribution by Strategy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/cost_distributions.png', dpi=150)
plt.show()

## 4. Savings Analysis vs No-Overbooking Baseline

In [7]:
baseline_cost = summary.loc['No Overbooking (0)', 'Avg Total Cost']
print(f'Baseline (No Overbooking) avg daily cost: ₹{baseline_cost:.2f}')
print('\nSavings vs baseline:')
for strat, row in summary.iterrows():
    saving = baseline_cost - row['Avg Total Cost']
    saving_pct = saving / baseline_cost * 100
    print(f'  {strat:<30}: ₹{saving:+.2f}  ({saving_pct:+.1f}%)')

# Best strategy
best_strat = summary['Avg Total Cost'].idxmin()
best_cost  = summary.loc[best_strat, 'Avg Total Cost']
annual_saving = (baseline_cost - best_cost) * 250  # ~250 working days
print(f'\nRecommended strategy: {best_strat}')
print(f'Average daily cost   : ₹{best_cost:.2f}')
print(f'Estimated annual saving vs baseline: ₹{annual_saving:,.0f}')

Baseline (No Overbooking) avg daily cost: ₹928.35

Savings vs baseline:
  No Overbooking (0)            : ₹+0.00  (+0.0%)
  Fixed +1                      : ₹+107.01  (+11.5%)
  Fixed +2                      : ₹+210.57  (+22.7%)
  Fixed +3                      : ₹+296.53  (+31.9%)
  Fixed +4                      : ₹+401.67  (+43.3%)
  Fixed +5                      : ₹+494.24  (+53.2%)
  Risk-Based Dynamic            : ₹+522.07  (+56.2%)

Recommended strategy: Risk-Based Dynamic
Average daily cost   : ₹406.28
Estimated annual saving vs baseline: ₹130,518


## 5. Sensitivity Analysis – Cost Heatmap

In [8]:
print('Running sensitivity analysis (cost heatmap)...')
print('This varies slots per day and fixed overbooking level.')
matrix = cost_heatmap(daily_pools,
                       slot_range=range(15, 26),
                       extra_range=range(0, 7),
                       n_days=500)
print('Cost heatmap saved.')

Running sensitivity analysis (cost heatmap)...
This varies slots per day and fixed overbooking level.


Cost heatmap saved.


## 6. Risk-Based Dynamic Overbooking – Detailed Analysis

In [9]:
if 'Risk-Based Dynamic' in raw_results:
    df_dyn = raw_results['Risk-Based Dynamic']

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Bookings per day
    axes[0,0].hist(df_dyn['n_booked'], bins=30, color='steelblue', alpha=0.8)
    axes[0,0].axvline(SLOTS_PER_DAY, color='red', linestyle='--',
                       label=f'Base capacity ({SLOTS_PER_DAY})')
    axes[0,0].set_title('Booked Patients per Day')
    axes[0,0].set_xlabel('Count')
    axes[0,0].legend()

    # Shows per day
    axes[0,1].hist(df_dyn['n_showed'], bins=30, color='mediumseagreen', alpha=0.8)
    axes[0,1].axvline(SLOTS_PER_DAY, color='red', linestyle='--',
                       label=f'Capacity ({SLOTS_PER_DAY})')
    axes[0,1].set_title('Actual Shows per Day')
    axes[0,1].set_xlabel('Count')
    axes[0,1].legend()

    # Idle slots over time
    axes[1,0].plot(df_dyn['idle_slots'].rolling(100).mean(), color='coral',
                    linewidth=1.5)
    axes[1,0].set_title('Rolling Avg Idle Slots (window=100 days)')
    axes[1,0].set_xlabel('Simulation Day')
    axes[1,0].set_ylabel('Avg Idle Slots')

    # Utilisation distribution
    axes[1,1].hist(df_dyn['utilisation'] * 100, bins=30, color='orchid', alpha=0.8)
    axes[1,1].axvline(df_dyn['utilisation'].mean()*100, color='navy',
                       linestyle='--', linewidth=2,
                       label=f'Mean: {df_dyn["utilisation"].mean()*100:.1f}%')
    axes[1,1].set_title('Slot Utilisation Distribution')
    axes[1,1].set_xlabel('Utilisation (%)')
    axes[1,1].legend()

    plt.suptitle('Risk-Based Dynamic Overbooking – Detailed Analysis',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/dynamic_overbooking_detail.png', dpi=150)
    plt.show()

## 7. Summary

**Simulation setup:**
- 5,000 Monte Carlo replications per strategy
- Base capacity: 20 slots/day
- Cost model: idle slot (₹150), patient waiting (₹2/min), doctor overtime (₹5/min)

**Key finding:**  
Risk-Based Dynamic Overbooking outperforms both No Overbooking and Fixed Overbooking in terms of total daily cost, achieving higher slot utilisation while controlling overcrowding.

**Decision support:**  
The recommended policy for clinic managers is to use risk-based dynamic overbooking, where the number of extra bookings equals the round of the sum of individual no-show risk scores for that day's cohort.